# Isotonic Regression - Starter Notebook
Isotonic regression fits a non-decreasing (or non-increasing) piecewise constant function, useful for calibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import mean_squared_error

In [ ]:
np.random.seed(42)
n = 150
X = np.sort(np.random.uniform(0, 10, n))
# True monotone signal + noise
y_true = 0.5 * X + 2 * np.sin(X * 0.5)
y_noisy = y_true + np.random.randn(n) * 1.5

In [ ]:
# Fit isotonic, increasing=True (default) and linear regression
ir = IsotonicRegression(out_of_bounds='clip')
y_iso = ir.fit_transform(X, y_noisy)

lr = LinearRegression().fit(X[:, None], y_noisy)
y_lin = lr.predict(X[:, None])

mse_iso = mean_squared_error(y_true, y_iso)
mse_lin = mean_squared_error(y_true, y_lin)
print(f'Isotonic Regression MSE (vs true): {mse_iso:.4f}')
print(f'Linear Regression    MSE (vs true): {mse_lin:.4f}')

In [ ]:
plt.figure(figsize=(9, 4))
plt.scatter(X, y_noisy, color='lightgray', s=15, label='Noisy observations')
plt.plot(X, y_true,    color='black',    linewidth=1.5, linestyle='--', label='True signal')
plt.step(X, y_iso,     color='steelblue', linewidth=2, where='post', label='Isotonic fit')
plt.plot(X, y_lin,     color='darkorange', linewidth=1.5, label='Linear fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Isotonic Regression vs Linear Regression')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Probability calibration use-case
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xc, yc = make_classification(n_samples=2000, n_features=10, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=42)

# Train uncalibrated model
clf = LogisticRegression(max_iter=500)
clf.fit(Xc_tr, yc_tr)
uncal_probs = clf.predict_proba(Xc_te)[:, 1]

# Calibrate with Isotonic Regression
ir_cal = IsotonicRegression(out_of_bounds='clip')
cal_probs = ir_cal.fit_transform(uncal_probs, yc_te)
cal_probs = np.clip(cal_probs, 0, 1)

frac_pos_u, mean_pred_u = calibration_curve(yc_te, uncal_probs, n_bins=10)
frac_pos_c, mean_pred_c = calibration_curve(yc_te, cal_probs,   n_bins=10)

plt.figure(figsize=(6, 5))
plt.plot([0,1],[0,1], 'k--', label='Perfect calibration')
plt.plot(mean_pred_u, frac_pos_u, 's-', label='Uncalibrated')
plt.plot(mean_pred_c, frac_pos_c, 'o-', label='Isotonic calibrated')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve: Isotonic Regression')
plt.legend()
plt.tight_layout()
plt.show()